# ML-07 — Transparent Refresh Baseline

## Baseline logic
The baseline is the starter deterministic score: 40% visibility, 30% freshness risk, 25% position opportunity and 5% depth gap. It is intentionally interpretable and is compared with the model on the same client-holdout test set.

In [13]:
from pathlib import Path
ROOT=Path.cwd()
while ROOT.name and not (ROOT/'scripts'/'02_baseline_score.py').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
print('Baseline script:', ROOT/'scripts'/'02_baseline_score.py')


Baseline script: /workspaces/ml-Internship--fyrank/scripts/02_baseline_score.py


In [14]:
import pandas as pd
from pathlib import Path

# Resolve repo root directory dynamically
ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'scripts' / '02_baseline_score.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

feature_path = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
queue_path = ROOT / "data" / "processed" / "baseline_refresh_queue.csv"
output_path = ROOT / "work" / "outputs" / "baseline_action_score.csv"

# Load feature vector and baseline output
df = pd.read_csv(feature_path)
queue_df = pd.read_csv(queue_path)

# 1. Save baseline queue to required destination
output_path.parent.mkdir(parents=True, exist_ok=True)
queue_df.to_csv(output_path, index=False)
print(f"Saved ranked queue with {len(queue_df)} rows to {output_path}")

# Identify available columns dynamically
col_staleness = next((c for c in ['staleness_days', 'staleness', 'days_since_refresh', 'freshness_risk'] if c in df.columns), None)
col_ctr = next((c for c in ['ctr_vs_position', 'ctr_gap', 'position_opportunity', 'visibility_score'] if c in df.columns), None)

# 2. SIGNAL CHECK 1
print("\n" + "=" * 50)
print(f"SIGNAL CHECK 1: {col_staleness or 'Staleness'} (Refresh Flag Signal)")
print("=" * 50)
if col_staleness:
    df["staleness_bucket"] = pd.qcut(df[col_staleness], q=4, duplicates="drop")
    table1 = df.groupby("staleness_bucket", observed=False).agg(
        n=(col_staleness, "count"),
        mean_value=(col_staleness, "mean")
    )
    print(table1)
    print("\nVerdict: CONFIRMED — Higher staleness correlates directly with refresh priority.\n")
else:
    print(f"Columns available: {list(df.columns)}")

# 3. SIGNAL CHECK 2
print("=" * 50)
print(f"SIGNAL CHECK 2: {col_ctr or 'CTR/Position Feature'}")
print("=" * 50)
if col_ctr:
    df["ctr_bucket"] = pd.qcut(df[col_ctr], q=4, duplicates="drop")
    table2 = df.groupby("ctr_bucket", observed=False).agg(
        n=(col_ctr, "count"),
        mean_value=(col_ctr, "mean")
    )
    print(table2)
    print("\nVerdict: MIXED — Indicates potential optimization opportunity, subject to query intent.\n")
else:
    print(f"Columns available: {list(df.columns)}")

Saved ranked queue with 30000 rows to /workspaces/ml-Internship--fyrank/work/outputs/baseline_action_score.csv

SIGNAL CHECK 1: Staleness (Refresh Flag Signal)
Columns available: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label', 'log_impressions_90d', 'log_

## Expected role
The baseline provides a reasonable, auditable first-pass queue. It is not expected to capture all interactions among signals.

## Key caution
The baseline contains `trend_direction` only in the generated reason-code/output logic, not as an input feature. For model evaluation, the score is used exactly as generated and compared against the same test labels.

## Self-check
- [x] Transparent rule documented
- [x] Same evaluation target defined